# MiniGPT learns from a teacher: follow along

This notebook goes with my post [MiniGPT (Part 6)](https://haddley.github.io/posts/minigpt5/). It looks at a real teacher's wheel of chances, works out how different two wheels are, and trains a small MiniGPT with and without GPT-2 as its teacher.

**It needs a Mac with Apple Silicon**, because MLX only runs there. Open it in Jupyter from a clone of [minigpt-series](https://github.com/Haddley/minigpt-series), with the packages from its `requirements.txt` installed. The first run downloads GPT-2 (about 500 MB) from Hugging Face.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))
from notebook_setup import setup
run = setup("part6-distillation")

working in part6-distillation


## 1. The answer, and the teacher's wheel

After "Tim gave his dog a", the guessing game only knows the one right answer. GPT-2, as a teacher, has a chance for every one of its 50,257 pieces.

In [2]:
import mlx.core as mx
import mlx.nn as nn
from mlx_lm import load

teacher, tokenizer = load("openai-community/gpt2")
ids = tokenizer.encode("Tim gave his dog a")
logits = teacher(mx.array([ids]))[0, -1]
wheel = mx.softmax(logits)
top = mx.argsort(-wheel)[:8].tolist()
print("the teacher's wheel, biggest slices:")
for i in top:
    print(f"  {tokenizer.decode([i])!r:12} {wheel[i].item():6.1%}")
print("\nthe one-hot answer, if the story said ' bone':")
print(f"  {' bone'!r:12} 100.0%, and every other piece 0%")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

the teacher's wheel, biggest slices:
  ' hug'         6.2%
  ' big'         2.1%
  ' good'        2.0%
  ' "'           1.6%
  ' ride'        1.5%
  ' run'         1.3%
  ' leash'       1.3%
  ' warm'        1.3%

the one-hot answer, if the story said ' bone':
  ' bone'      100.0%, and every other piece 0%


## 2. How different are two wheels?

KL divergence is 0 for identical wheels, and grows as they differ. Softening both with a temperature of 2 first, as the post does, makes the near misses count.

In [3]:
def kl(teacher_logits, student_logits, temp=2.0):
    t = nn.log_softmax(teacher_logits / temp)
    s = nn.log_softmax(student_logits / temp)
    return (mx.exp(t) * (t - s)).sum().item()

even = mx.zeros_like(logits)                  # a student who knows nothing: an even wheel
print(f"teacher against itself:      {kl(logits, logits):.4f}")
print(f"teacher against an even wheel: {kl(logits, even):.4f}")
print(f"softened wheel's top slice: {mx.softmax(logits / 2)[top[0]].item():.1%}, against {wheel[top[0]].item():.1%} unsoftened")

teacher against itself:      0.0000
teacher against an even wheel: 1.4974
softened wheel's top slice: 0.4%, against 6.2% unsoftened


## 3. Train with and without a teacher

The same 30-million-number student, 3,000 steps each, on Part 3's stories with GPT-2's pieces. On my Mac Studio, 3,000 steps took about 13 minutes without a teacher, and 25 minutes with GPT-2 as the teacher.

In [4]:
run("train_distill.py", "--tag", "baseline", "--teacher", "none", "--alpha", "1.0", "--iters", "3000")
run("train_distill.py", "--tag", "gpt2", "--teacher", "gpt2", "--alpha", "0.5", "--iters", "3000")

tag baseline  alpha 1.0  temp 2.0  student params 30,035,328
step     0  train 11.2836  val 11.2850  val bpb 4.0028
step   300  train 3.4931  val 3.5105  val bpb 1.2452
step   600  train 2.9319  val 3.0099  val bpb 1.0676
step   900  train 2.6582  val 2.7556  val bpb 0.9774
step  1200  train 2.4799  val 2.5369  val bpb 0.8998
step  1500  train 2.3187  val 2.4343  val bpb 0.8634
step  1800  train 2.2238  val 2.3205  val bpb 0.8231
step  2100  train 2.1515  val 2.2646  val bpb 0.8033
step  2400  train 2.0801  val 2.1590  val bpb 0.7658
step  2700  train 2.0556  val 2.1481  val bpb 0.7619
step  3000  train 2.0094  val 2.1297  val bpb 0.7554
done in 11.47 min  best val 2.1297  peak 4.21 GB



tag gpt2  alpha 0.5  temp 2.0  student params 30,035,328
teacher openai-community/gpt2 loaded (mlx-lm) and frozen
step     0  train 11.2836  val 11.2850  val bpb 4.0028
step   300  train 3.5415  val 3.5559  val bpb 1.2613
step   600  train 2.9252  val 2.9911  val bpb 1.0609
step   900  train 2.6449  val 2.7232  val bpb 0.9659
step  1200  train 2.4590  val 2.4941  val bpb 0.8847
step  1500  train 2.3095  val 2.3929  val bpb 0.8488
step  1800  train 2.2229  val 2.2869  val bpb 0.8112
step  2100  train 2.1520  val 2.2307  val bpb 0.7912
step  2400  train 2.0918  val 2.1420  val bpb 0.7598
step  2700  train 2.0675  val 2.1302  val bpb 0.7556
step  3000  train 2.0288  val 2.1050  val bpb 0.7467
done in 22.46 min  best val 2.1050  peak 10.39 GB

Fetching 14 files: 100%|██████████| 14/14 [00:00<00:00, 1906.81it/s]



In [5]:
import json
for tag in ("baseline", "gpt2"):
    h = json.load(open(f"runs/history_{tag}.json"))
    best = min(h["history"], key=lambda p: p["val"])
    print(f"{tag:9} best bits per byte {best['bpb']:.4f}, peak memory {h['peak_gb']:.1f} GB, {h['elapsed_sec'] / 60:.1f} minutes")

baseline  best bits per byte 0.7554, peak memory 4.2 GB, 11.5 minutes
gpt2      best bits per byte 0.7467, peak memory 10.4 GB, 22.5 minutes
